<a href="https://colab.research.google.com/github/hieunh1990/brave-api/blob/staging/demo/demo_colab_remote_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DotsOCR vLLM Openai API Compatible server

In [1]:
!pip install pyngrok
!ngrok authtoken  # Get this from https://dashboard.ngrok.com/

ERROR:  accepts 1 arg(s), received 0


In [2]:
!conda create -n dots_ocr python=3.12
!conda activate dots_ocr

!git clone https://github.com/rednote-hilab/dots.ocr.git

/bin/bash: line 1: conda: command not found
/bin/bash: line 1: conda: command not found
Cloning into 'dots.ocr'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 199 (delta 45), reused 42 (delta 32), pack-reused 131 (from 1)
Receiving objects: 100% (199/199), 35.84 MiB | 17.64 MiB/s, done.
Resolving deltas: 100% (78/78), done.


In [3]:
cd /content/dots.ocr

/content/dots.ocr


In [ ]:
# Install pytorch, see https://pytorch.org/get-started/previous-versions/ for your cuda version
!pip install torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -e .

Looking in indexes: https://download.pytorch.org/whl/cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 GB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.6/609.6 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 726.9/726.9 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 MB 5.4 MB/s eta 0:00

In [ ]:
!python3 tools/download_model.py

In [ ]:
import os
from pathlib import Path

# Set up model path (using a directory in Colab's temporary storage)
hf_model_path = "./weights/DotsOCR"
os.environ["hf_model_path"] = hf_model_path

# Create directory if it doesn't exist
Path(hf_model_path).mkdir(parents=True, exist_ok=True)

# Add to PYTHONPATH
os.environ["PYTHONPATH"] = f"{os.path.dirname(hf_model_path)}:{os.environ.get('PYTHONPATH', '')}"

# Install required packages
!pip install vllm transformers

# Modify vllm import (this is a workaround - may need adjustment based on vllm version)
try:
    vllm_path = !which vllm
    if vllm_path:
        vllm_path = vllm_path[0]
        !sed -i '/^from vllm\.entrypoints\.cli\.main import main$/a from DotsOCR import modeling_dots_ocr_vllm' {vllm_path}
except:
    print("Could not automatically modify vllm imports. You may need to do this manually.")

In [ ]:
from pyngrok import ngrok
public_url = ngrok.connect(8000, bind_tls=True)  # Adjust port if needed
print("Public URL:", public_url)

In [ ]:
!CUDA_VISIBLE_DEVICES=0 vllm serve ./weights/DotsOCR --tensor-parallel-size 1 --gpu-memory-utilization 0.95  --chat-template-content-format string --served-model-name model --trust-remote-code

2025-08-07 20:57:52.107021: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-07 20:57:52.125111: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754600272.146783   10516 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754600272.153513   10516 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754600272.170115   10516 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 